# 📄 Descripción general del código

 Este script entrena un modelo de clasificación binaria utilizando XLM-RoBERTa
 para detectar si hay una relación entre dos oraciones (por ejemplo, Claim–Evidence).
 El flujo general incluye:

 1. Instalación de bibliotecas necesarias (Transformers, Datasets, scikit-learn)
 2. Carga de un dataset desde Google Drive en formato CSV
 3. Conversión de etiquetas multicategoría a etiquetas binarias (0 = sin relación, 1 = con relación)
 4. Balanceo de clases mediante undersampling
 5. División del dataset en entrenamiento y validación
 6. Tokenización con el modelo "xlm-roberta-large"
 7. Entrenamiento del modelo con Hugging Face Trainer
 8. Evaluación usando métricas como Accuracy y F1
 9. Guardado del modelo y del tokenizador en Google Drive

 Ideal para tareas de clasificación de pares de frases en español (o multilingüe).

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# ✅ Instalación de bibliotecas

!pip install transformers datasets scikit-learn -q

In [ ]:
# ✅ Importaciones básicas

import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# ✅ Montar Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ Cargar archivo CSV

data = pd.read_csv('data/cross_encoder_train_data.csv')
data['label'] = data['label'].astype(int)

In [ ]:
# ✅ Crear etiquetas binarias (0 vs relación existente)

data['binary_label'] = data['label'].apply(lambda x: 0 if x == 0 else 1)

In [ ]:
# ✅ Balanceo de clases (undersampling 1:1)

rel_df = data[data['binary_label'] == 1]
nonrel_df = data[data['binary_label'] == 0]
nonrel_sample = resample(nonrel_df, replace=False, n_samples=len(rel_df), random_state=42)

balanced_data = pd.concat([rel_df, nonrel_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# ✅ train/validation split
# ✅ División en conjunto de entrenamiento y validación

train_df, val_df = train_test_split(balanced_data, test_size=0.1, random_state=42, stratify=balanced_data['binary_label'])

In [ ]:
# ✅ Conversión a Hugging Face Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
print(data.columns)

Index(['sentence1', 'sentence2', 'label', 'binary_label'], dtype='object')


In [ ]:
# ✅ Tokenizer
# ✅ Cargar el tokenizador

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")

# ✅ Función de tokenización
def tokenize(batch):
    return tokenizer(batch['sentence1'], batch['sentence2'], truncation=True, padding='max_length', max_length=128)

# ✅ Aplicar tokenización
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# ✅ Renombrar columna de etiquetas y configurar formato para PyTorch
train_dataset = train_dataset.rename_column("binary_label", "labels")
val_dataset = val_dataset.rename_column("binary_label", "labels")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# ✅ Cargar el modelo (clasificación binaria)

model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-large", num_labels=2)

In [ ]:
# ✅ Definir función de métricas

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [ ]:
# ✅ Configuración del entrenamiento

training_args = TrainingArguments(
    output_dir="models/results_support_matching",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=20,
)

In [ ]:
# ✅ Definir el Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# ✅ Iniciar el entrenamiento

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.687500,0.692903,0.500000,0.000000
2,0.656600,0.634083,0.656604,0.558252
3,0.589100,0.512167,0.766038,0.767790
4,0.438600,0.460480,0.805660,0.808194
5,0.343900,0.511495,0.818868,0.833910
6,0.302300,0.486813,0.807547,0.806084
7,0.326400,0.523358,0.815094,0.816479
8,0.331400,0.536176,0.811321,0.809886


TrainOutput(global_step=4776, training_loss=0.495632839262785, metrics={'train_runtime': 1022.4132, 'train_samples_per_second': 37.323, 'train_steps_per_second': 4.671, 'total_flos': 8890625206149120.0, 'train_loss': 0.495632839262785, 'epoch': 8.0})

In [ ]:
# ✅ Guardar el modelo y el tokenizador

model.save_pretrained("models/support_matching_xlmr")
tokenizer.save_pretrained("models/support_matching_xlmr")

In [ ]:
import os
# ✅ Mostrar archivos del modelo guardado

model_path = "models/support_matching_xlmr"
print(os.listdir(model_path))